# REACTO — Bayesian optimisation of the DLR phosphorylation

A condensed, self-contained reference implementation of the algorithm behind
the **catalyst-loading campaign**. It reproduces, step by step, what REACTO
does between one experiment and the next, and it runs on the experiments
actually recorded during the campaign.

REACTO is a Bayesian optimisation framework for chemical reactions, built on
[BoFire](https://github.com/experimental-design/bofire) and
[BoTorch](https://botorch.org). A campaign is a closed loop:

1. the reaction space and the objective are declared once, as a *domain*;
2. a space-filling **initial design** is drawn inside that space;
3. the experiments performed so far are used to fit a Gaussian-process
   surrogate of the objective;
4. the **qLogNEI** acquisition function is maximised over the reaction space,
   giving the next set of conditions to run;
5. the chemist runs it, records the yield, and the loop returns to step 3.

Nothing is batched and nothing is simulated: one experiment is proposed, one
experiment is performed, the surrogate is refitted.

## The optimisation problem

| | |
|---|---|
| Parameters | `[DMIPP]` 1.0–6.7 equiv (0.2 steps) · `BuOH` 1.0–10.8 equiv (0.2) · `Cat. Loading` 5 / 7.5 / 10 mol% · `Temperature` 25–75 °C (5) · `Res. Time` 0.5–2.5 min (0.5) |
| Objective | maximise `Yield` (%) |
| Process constraint | `[DMIPP]` ≤ `BuOH` |
| Initial design | 10 points, *k*-means |
| Acquisition | qLogNEI, one experiment at a time |

The campaign reported here is 10 initial experiments followed by 9 proposals.

## 0. Setup

Python 3.11, `bofire==0.3.1`, `botorch==0.17.0`, `gpytorch==1.15.2`,
`torch==2.10.0`, `scikit-learn==1.8.0`.

In [1]:
import warnings
from itertools import product

import numpy as np
import pandas as pd

import bofire.strategies.api as strategies
from bofire.data_models.api import Domain, Inputs, Outputs, Constraints
from bofire.data_models.features.api import (
    ContinuousInput, DiscreteInput, CategoricalInput, ContinuousOutput,
)
from bofire.data_models.objectives.api import MaximizeObjective, MinimizeObjective
from bofire.data_models.constraints.api import LinearInequalityConstraint
from bofire.data_models.acquisition_functions.api import qLogNEI
from bofire.data_models.strategies.api import SoboStrategy

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)

N_INITIAL    = 10     # size of the initial design
BATCH_SIZE   = 1      # one experiment proposed at a time
RANDOM_STATE = 42     # seed of the initial design

## 1. The reaction space

The domain is the single declarative object that carries the whole problem:
what can be varied and over which values, what is being optimised and in which
direction, and which combinations of parameters are forbidden.

Two things about this particular space are worth making explicit in a methods
section.

**Every parameter is discrete.** Concentration, alcohol loading and temperature
are continuous quantities, but they are declared with an increment — 0.2 equiv,
0.2 equiv, 5 °C — and REACTO expands each into an explicit grid of levels.
Catalyst loading and residence time are discrete by nature. The optimiser
therefore never proposes conditions a chemist cannot set on the rig; the price
is that the search is over a finite grid of about 2.4 × 10⁵ points rather than
over a continuum. A declared upper bound that does not fall on the increment is
not itself a level: `[DMIPP]` is declared to 6.7 equiv but its grid, stepping by
0.2 from 1.0, stops at 6.6.

**`[DMIPP] ≤ BuOH` is known before any experiment is run.** It is a process
constraint, written as the linear inequality `1·[DMIPP] + (−1)·BuOH ≤ 0`, and
it is enforced exactly — both when the initial design is drawn and when the
acquisition function is maximised. Conditions violating it are never proposed,
so no experiment is spent discovering that they are impractical.

The objective declares a direction and the bounds of its useful range, here the
full 0–100 % of a yield.

In [2]:
def grid(lower, upper, step):
    # Expand a continuous range into the levels REACTO actually searches over.
    values = np.arange(lower, upper + step / 2, step)
    return [round(float(v), 6) for v in values]


domain = Domain(
    inputs=Inputs(features=[
        DiscreteInput(key="[DMIPP]",      values=grid(1.0, 6.7, 0.2),   unit="equiv"),
        DiscreteInput(key="BuOH",         values=grid(1.0, 10.8, 0.2),  unit="equiv"),
        DiscreteInput(key="Cat. Loading", values=[5.0, 7.5, 10.0],      unit="mol%"),
        DiscreteInput(key="Temperature",  values=grid(25.0, 75.0, 5.0), unit="degC"),
        DiscreteInput(key="Res. Time",    values=[0.5, 1.0, 1.5, 2.0, 2.5], unit="min"),
    ]),
    outputs=Outputs(features=[
        ContinuousOutput(key="Yield", objective=MaximizeObjective(w=1.0, bounds=(0.0, 100.0))),
    ]),
    constraints=Constraints(constraints=[
        # [DMIPP] <= BuOH  ->  1*[DMIPP] + (-1)*BuOH <= 0
        LinearInequalityConstraint(
            features=["[DMIPP]", "BuOH"], coefficients=[1.0, -1.0], rhs=0.0,
        ),
    ]),
)

PARAMETERS = [f.key for f in domain.inputs.features]
OBJECTIVE  = domain.outputs.features[0].key

levels = {f.key: list(f.values) for f in domain.inputs.features}
print(f"{int(np.prod([len(v) for v in levels.values()])):,} candidate conditions before constraints")
pd.Series({k: f"{len(v)} levels, {min(v)} to {max(v)}" for k, v in levels.items()},
          name="reaction space").to_frame()

239,250 candidate conditions before constraints


,reaction space
[DMIPP],"29 levels, 1.0 to 6.6"
BuOH,"50 levels, 1.0 to 10.8"
Cat. Loading,"3 levels, 5.0 to 10.0"
Temperature,"11 levels, 25.0 to 75.0"
Res. Time,"5 levels, 0.5 to 2.5"


## 2. Initial design

Bayesian optimisation needs something to learn from before it can propose
anything. REACTO draws that first set of conditions with a space-filling
design — Latin hypercube, random, Sobol, or the constrained *k*-means used
here, which is the approach of Shields *et al.* and of Kaneko.

The grid is enumerated, points violating the process constraint are discarded,
the surviving points are min–max scaled, and *k*-means is run with as many
clusters as experiments wanted; the feasible point nearest each centroid is
selected. The result covers the reaction space evenly *within the feasible
region*, which a design drawn over the box and filtered afterwards would not.

The grid here holds more points than are worth enumerating exhaustively, so a
random subset of 10⁵ is drawn first and clustered. The design is seeded and is
therefore reproducible.

In [3]:
def is_feasible(domain, points):
    # Rows of `points` satisfying every linear constraint of the domain.
    mask = pd.Series(True, index=points.index)
    if domain.constraints is None:
        return mask
    for con in domain.constraints.constraints:
        if not isinstance(con, LinearInequalityConstraint):
            continue
        lhs = sum(coefficient * points[key].astype(float)
                  for key, coefficient in zip(con.features, con.coefficients))
        mask &= lhs <= float(con.rhs) + 1e-6
    return mask


def kmeans_initial_design(domain, n_points, random_state=RANDOM_STATE, pool_size=100_000):
    # Space-filling design by k-means over the feasible region of the grid.
    keys = [f.key for f in domain.inputs.features]
    values = {}
    for feat in domain.inputs.features:
        if isinstance(feat, DiscreteInput):
            values[feat.key] = list(feat.values)
        elif isinstance(feat, ContinuousInput):
            lower, upper = feat.bounds                       # a continuous parameter is
            values[feat.key] = grid(lower, upper, (upper - lower) / 20)   # gridded for this step only
        elif isinstance(feat, CategoricalInput):
            values[feat.key] = list(feat.categories)

    # 1. Candidate pool: the whole grid, or a random subset when it is too large.
    total = int(np.prod([len(values[k]) for k in keys]))
    if total > pool_size:
        rng = np.random.default_rng(random_state)
        pool = pd.DataFrame({k: rng.choice(values[k], size=pool_size) for k in keys})
    else:
        pool = pd.DataFrame(list(product(*[values[k] for k in keys])), columns=keys)

    # 2. Keep only what the process constraint allows.
    pool = pool[is_feasible(domain, pool)].reset_index(drop=True)
    if len(pool) == 0:
        raise ValueError("no feasible point: check the constraints and the bounds")
    if n_points >= len(pool):
        return pool

    # 3. Cluster the feasible region, categoricals encoded as integers.
    encoded = pool.copy()
    for key in keys:
        if encoded[key].dtype == object:
            encoded[key] = pd.Categorical(encoded[key], categories=sorted(set(pool[key]))).codes
    scaled = MinMaxScaler().fit_transform(encoded[keys].astype(float).to_numpy())
    centroids = KMeans(n_clusters=n_points, random_state=random_state, n_init=20).fit(scaled)

    # 4. One experiment per cluster: the feasible point nearest its centroid.
    chosen = []
    for centroid in centroids.cluster_centers_:
        order = np.argsort(np.linalg.norm(scaled - centroid, axis=1))
        chosen.append(next(i for i in order if i not in chosen))
    return pool.iloc[chosen].reset_index(drop=True)


kmeans_initial_design(domain, N_INITIAL)

,[DMIPP],BuOH,Cat. Loading,Temperature,Res. Time
0,2.0,5.8,7.5,40.0,1.0
1,3.0,6.6,10.0,35.0,1.0
2,2.2,6.0,10.0,55.0,2.0
3,5.2,8.2,5.0,40.0,1.0
4,5.0,8.4,10.0,40.0,2.0
5,3.4,7.0,7.5,65.0,1.0
6,2.8,7.2,5.0,35.0,2.0
7,3.8,7.6,10.0,65.0,1.0
8,2.0,5.8,7.5,60.0,2.0
9,5.0,8.2,5.0,60.0,2.0


## 3. Surrogate and acquisition function

Each proposal is made in two steps: a surrogate of the yield is fitted to
everything measured so far, and the conditions maximising an acquisition
function computed from that surrogate are returned.

**The surrogate** is a Gaussian process: a Matérn-5/2 kernel with automatic
relevance determination — one length scale per parameter, so the fit itself
reports which parameters the yield is sensitive to — over inputs scaled to the
unit cube, with a standardised output and a fitted noise term. Its
hyperparameters are set by maximising the marginal likelihood. What it returns
at any set of conditions is not a number but a posterior: a predicted yield and
the uncertainty on that prediction.

**The acquisition function** is qLogNEI, the log-transformed noisy expected
improvement. Expected improvement scores a candidate by how much it is expected
to beat the best yield obtained so far, under the posterior; the *noisy*
variant treats the recorded yields as noisy realisations rather than as ground
truth, which is the honest assumption for a bench measurement; the *log*
formulation is the numerically stable reparametrisation that keeps gradients
informative where the probability of improvement is vanishingly small.

This is what balances exploration against exploitation without either being
specified by hand: a candidate scores well either because its predicted yield
is high, or because the posterior there is wide enough that it might be. The
maximisation runs over the grid of §1, subject to the process constraint.

In [4]:
def suggest_next_experiment(domain, experiments, q=BATCH_SIZE, acquisition=None, seed=None):
    # Fit the surrogate on everything measured so far and propose the next conditions.
    data_model = SoboStrategy(
        domain=domain,
        acquisition_function=acquisition or qLogNEI(),
        **({} if seed is None else {"seed": seed}),
    )
    strategy = strategies.map(data_model)
    strategy.tell(experiments=experiments)           # everything recorded so far
    return strategy.ask(candidate_count=q), strategy # ... and the conditions to run next

## 4. The campaign

The experiments recorded during the campaign: 10 from the initial design of §2,
then 9 proposed one at a time by §3. `Yield` is in per cent.

In [5]:
EXPERIMENTS = pd.DataFrame([
    # type, [DMIPP], BuOH, Cat. Loading, Temperature, Res. Time, Yield
    ("Init", 2.2,  6.0,   5.0, 50.0, 1.0, 16.0),
    ("Init", 5.0,  8.6,   7.5, 60.0, 2.0, 50.0),
    ("Init", 3.4,  7.0,   5.0, 35.0, 2.0, 16.0),
    ("Init", 5.2,  8.4,   7.5, 40.0, 1.0, 19.0),
    ("Init", 5.0,  8.4,   5.0, 50.0, 1.0, 15.0),
    ("Init", 3.0,  7.4,  10.0, 65.0, 1.0, 69.0),
    ("Init", 3.4,  7.0,   5.0, 65.0, 2.0, 30.0),
    ("Init", 2.2,  6.0,   7.5, 60.0, 2.0, 61.0),
    ("Init", 3.2,  7.4,  10.0, 35.0, 2.0, 36.0),
    ("Init", 2.2,  5.8,   7.5, 35.0, 1.0, 16.0),
    ("BO",   1.8,  6.0,  10.0, 60.0, 2.0, 72.0),
    ("BO",   1.0,  3.8,  10.0, 70.0, 2.0, 85.0),
    ("BO",   1.0,  1.0,  10.0, 75.0, 2.5, 58.0),
    ("BO",   1.0,  5.0,  10.0, 75.0, 1.5, 77.0),
    ("BO",   5.0,  5.0,  10.0, 70.0, 2.5, 55.0),
    ("BO",   1.0,  1.0,  10.0, 70.0, 2.0, 73.0),
    ("BO",   1.0,  7.6,  10.0, 70.0, 1.5, 53.0),
    ("BO",   1.6,  3.4,  10.0, 65.0, 1.5, 88.0),
    ("BO",   2.6,  3.2,  10.0, 70.0, 1.0, 68.0),
], columns=["Point type"] + PARAMETERS + [OBJECTIVE])

best = EXPERIMENTS.loc[EXPERIMENTS[OBJECTIVE].idxmax()]
print(f"{len(EXPERIMENTS)} experiments: "
      f"{int((EXPERIMENTS['Point type'] == 'Init').sum())} initial design, "
      f"{int((EXPERIMENTS['Point type'] == 'BO').sum())} proposed")
print(f"best of the initial design : {EXPERIMENTS[EXPERIMENTS['Point type'] == 'Init'][OBJECTIVE].max():.0f} %")
print(f"best of the campaign       : {best[OBJECTIVE]:.0f} % at "
      + ", ".join(f"{k}={best[k]:g}" for k in PARAMETERS))
EXPERIMENTS

19 experiments: 10 initial design, 9 proposed
best of the initial design : 69 %
best of the campaign       : 88 % at [DMIPP]=1.6, BuOH=3.4, Cat. Loading=10, Temperature=65, Res. Time=1.5


,Point type,[DMIPP],BuOH,Cat. Loading,Temperature,Res. Time,Yield
0,Init,2.2,6.0,5.0,50.0,1.0,16.0
1,Init,5.0,8.6,7.5,60.0,2.0,50.0
2,Init,3.4,7.0,5.0,35.0,2.0,16.0
3,Init,5.2,8.4,7.5,40.0,1.0,19.0
4,Init,5.0,8.4,5.0,50.0,1.0,15.0
5,Init,3.0,7.4,10.0,65.0,1.0,69.0
6,Init,3.4,7.0,5.0,65.0,2.0,30.0
7,Init,2.2,6.0,7.5,60.0,2.0,61.0
8,Init,3.2,7.4,10.0,35.0,2.0,36.0
9,Init,2.2,5.8,7.5,35.0,1.0,16.0


### What the campaign did

Plotting the best yield reached against the number of experiments separates the
two phases: the initial design maps the space, the proposals exploit it. The
running maximum is the quantity a campaign is judged on — how few experiments
were needed to reach a given yield.

In [6]:
progress = EXPERIMENTS.assign(**{"Best so far": EXPERIMENTS[OBJECTIVE].cummax()})
progress.index = np.arange(1, len(progress) + 1)
progress[["Point type", OBJECTIVE, "Best so far"]].T

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
Point type,Init,Init,Init,Init,Init,Init,Init,Init,Init,Init,BO,BO,BO,BO,BO,BO,BO,BO,BO
Yield,16.0,50.0,16.0,19.0,15.0,69.0,30.0,61.0,36.0,16.0,72.0,85.0,58.0,77.0,55.0,73.0,53.0,88.0,68.0
Best so far,16.0,50.0,50.0,50.0,50.0,69.0,69.0,69.0,69.0,69.0,72.0,85.0,85.0,85.0,85.0,85.0,85.0,88.0,88.0


## 5. Proposing the next experiment

The surrogate is refitted on all 19 experiments and the acquisition function is
maximised. Alongside the conditions, the strategy reports the surrogate's
posterior there: the predicted yield `Yield_pred` and its standard deviation
`Yield_sd`.

In [7]:
proposal, strategy = suggest_next_experiment(domain, EXPERIMENTS[PARAMETERS + [OBJECTIVE]])
proposal

,BuOH,Cat. Loading,Res. Time,Temperature,[DMIPP],Yield_pred,Yield_sd,Yield_des
0,3.0,10.0,2.0,60.0,1.0,86.551731,5.994034,0.865517


These are the conditions that were in fact recorded as the next experiment of
the campaign — `[DMIPP]` 1.0 equiv, `BuOH` 3.0 equiv, 10 mol% catalyst, 60 °C,
2.0 min — which is the check that this condensed implementation is the same
procedure and not merely a similar one. Repeated unseeded runs return it
unchanged: the surrogate is confident enough here that the arg-max does not
move with the Monte-Carlo draw.

### Reading the proposal

A proposal with a high predicted yield and a narrow posterior is the surrogate
exploiting what it already knows; one with a wide posterior is it proposing an
experiment that would be informative whatever the outcome. Comparing the
prediction against the best yield recorded so far says which of the two this
is.

The same posterior can be queried anywhere in the reaction space, which is what
makes the fitted surrogate useful beyond the next proposal — for instance to
map the predicted yield along one parameter with the others held at their
optimal values.

In [8]:
def predicted_profile(strategy, reference, parameter, levels_of):
    # Posterior mean and sd along one parameter, the others held at `reference`.
    scan = pd.DataFrame([{**reference, parameter: value} for value in levels_of[parameter]])
    prediction = strategy.predict(scan[PARAMETERS])
    return pd.DataFrame({
        parameter: scan[parameter],
        "predicted yield": prediction[f"{OBJECTIVE}_pred"].round(1),
        "sd":              prediction[f"{OBJECTIVE}_sd"].round(1),
    }).set_index(parameter)


reference = best[PARAMETERS].to_dict()   # the best conditions recorded so far
predicted_profile(strategy, reference, "Temperature", levels).T

Temperature,25.0,30.0,35.0,40.0,45.0,50.0,55.0,60.0,65.0,70.0,75.0
predicted yield,31.4,38.4,47.0,56.4,66.0,74.8,81.7,86.0,87.3,85.6,81.1
sd,17.3,15.8,14.3,12.5,10.5,8.3,5.9,3.8,2.6,2.9,4.3


## 6. The closed loop

Everything above is one turn of the loop. A campaign is that turn repeated: the
surrogate is refitted on every experiment recorded so far, a new proposal is
computed, the chemist performs it and records the yield. `run_experiment`
stands for the bench: it receives a set of conditions and returns the measured
yield.

Two properties are worth stating explicitly in a methods section. The surrogate
is refitted from scratch at each iteration, so a campaign is fully determined
by the set of experiments recorded — the order in which they were run does not
enter. And an experiment that disappoints is not wasted: a low yield constrains
the surrogate exactly as a high one does, and is what steers the next proposal
elsewhere.

In [9]:
def run_campaign(domain, experiments, run_experiment, n_iterations, seed=None):
    # Iterate propose -> perform -> record, returning the campaign so far.
    experiments = experiments.copy()
    for iteration in range(n_iterations):
        proposal, _ = suggest_next_experiment(
            domain, experiments[PARAMETERS + [OBJECTIVE]], q=1,
            seed=None if seed is None else seed + iteration,
        )
        conditions = proposal.iloc[0][PARAMETERS].to_dict()
        measured   = run_experiment(conditions)                 # <- the bench
        experiments = pd.concat(
            [experiments, pd.DataFrame([{**conditions, OBJECTIVE: measured}])],
            ignore_index=True,
        )
        print(f"[{iteration + 1:>2}/{n_iterations}] "
              + ", ".join(f"{k}={v:g}" for k, v in conditions.items())
              + f" -> {measured:.0f} %, best so far {experiments[OBJECTIVE].max():.0f} %")
    return experiments


# In this campaign run_experiment was a chemist at the bench. Substituting a
# response surface here turns the same loop into an in-silico benchmark.
#
# EXPERIMENTS = run_campaign(domain, EXPERIMENTS, run_experiment, n_iterations=10)

## Notes

**Discretisation.** Declaring an increment on a continuous parameter replaces it
with an explicit grid of levels. This is what keeps proposals settable on the
rig, and it is not free: the finer the increment the larger the search, and a
parameter whose true optimum falls between two levels can only be approached to
within half an increment.

**Reproducibility.** The strategy draws its own Monte-Carlo seed unless one is
passed, and neither `np.random.seed` nor `torch.manual_seed` reaches it. Two
runs on identical data will therefore differ wherever two candidate conditions
score closely, since the arg-max flips. Where they do not — as at the end of
this campaign, §5 — the proposal is stable without a seed. Pass `seed=` to
`suggest_next_experiment` to make a campaign replayable regardless; the one
reported in §4 was run without one.

**Cost.** One iteration is dominated by fitting the Gaussian process, which is
cubic in the number of experiments recorded. At the scale of a bench campaign —
tens to a few hundred experiments — a proposal takes seconds on a laptop CPU.

**Beyond one objective.** The same machinery extends to two objectives, e.g.
yield and space-time yield: one Gaussian process is fitted per objective and
qLogNEI is replaced by qLogNEHVI, the expected improvement of the hypervolume
dominated by the Pareto front. A threshold on an objective that is only known
once the experiment has been run — a minimum acceptable yield — is handled by
fitting an additional Gaussian process to the constraint and weighting each
candidate by the posterior probability that it is satisfied. Neither was used
in this campaign.

## References

1. Ament, S.; Daulton, S.; Eriksson, D.; Balandat, M.; Bakshy, E.
   *Unexpected Improvements to Expected Improvement for Bayesian Optimization.*
   NeurIPS **2023**. — qLogNEI
2. Balandat, M. *et al.* *BoTorch: A Framework for Efficient Monte-Carlo
   Bayesian Optimization.* NeurIPS **2020**.
3. Durholt, J. P. *et al.* *BoFire: Bayesian Optimization Framework Intended
   for Real Experiments.* **2024**, arXiv:2408.05040.
4. Shields, B. J. *et al.* *Bayesian reaction optimization as a tool for
   chemical synthesis.* *Nature* **2021**, *590*, 89–96.
5. Kaneko, H. *Sparse Sampling for Chemical Experiments.* *ACS Omega* **2022**,
   *7*, 47789–47795. — k-means initial design